# Пример аналитического notebook (ERGO MS)

Этот notebook демонстрирует работу с:
- **Django ORM** -- запросы к моделям проекта
- **SQLAlchemy** -- прямые SQL-запросы через `SqlAlchemyManager` / `DBConfig`
- **pandas** -- агрегация и анализ данных

> При использовании ядра **Django Kernel (ERGO MS)** все модели и утилиты импортированы автоматически.
> Пользователи — модель **`ErgoUser`** (`AUTH_USER_MODEL`), не `django.contrib.auth.models.User`.

## 1. Django ORM -- базовые запросы

Все модели проекта уже доступны по имени класса (автоимпорт ядра).

In [ ]:
# В ERGO MS AUTH_USER_MODEL = cms_adp.ErgoUser; ядро уже импортирует ErgoUser
total_users = ErgoUser.objects.count()
active_users = ErgoUser.objects.filter(is_active=True).count()
staff_users = ErgoUser.objects.filter(is_staff=True).count()

print(f'Всего пользователей: {total_users}')
print(f'Активных: {active_users}')
print(f'Сотрудников (is_staff): {staff_users}')

In [ ]:
# Последние зарегистрированные пользователи
recent_users = ErgoUser.objects.order_by('-date_joined')[:10].values(
    'public_id', 'username', 'email', 'date_joined', 'last_login', 'is_active'
)

df_users = pd.DataFrame(recent_users)
df_users

### ORM-агрегации

In [ ]:
from django.db.models.functions import TruncMonth

# Регистрации по месяцам
registrations = (
    ErgoUser.objects
    .annotate(month=TruncMonth('date_joined'))
    .values('month')
    .annotate(count=Count('id'))
    .order_by('month')
)

df_reg = pd.DataFrame(registrations)
if not df_reg.empty:
    df_reg['month'] = pd.to_datetime(df_reg['month'])
    df_reg.set_index('month', inplace=True)
    df_reg.plot(kind='bar', title='Регистрации по месяцам', figsize=(10, 4))
else:
    print('Нет данных о регистрациях')

### Связанные модели -- профили и роли

In [ ]:
# Профили пользователей с информацией о ролях
profiles = (
    UserProfile.objects
    .select_related('user')
    .values('user__username', 'country', 'city', 'language', 'timezone', 'created_at')
    [:20]
)

df_profiles = pd.DataFrame(profiles)
df_profiles

In [ ]:
# Распределение пользователей по ролям
roles_dist = (
    UserRole.objects
    .filter(is_active=True)
    .values('role__name')
    .annotate(user_count=Count('user', distinct=True))
    .order_by('-user_count')
)

df_roles = pd.DataFrame(roles_dist)
if not df_roles.empty:
    print('Распределение пользователей по ролям:')
    print(df_roles.to_string(index=False))
else:
    print('Нет данных о ролях')

## 2. SQLAlchemy -- прямые SQL-запросы

Используем `DBConfig` и `SqlAlchemyManager` из ядра для работы с БД напрямую через SQLAlchemy engine.

In [ ]:
from src.core.utils.database.dbconfig import DBConfig
from src.core.utils.database import SqlAlchemyManager

config = DBConfig('default')
sa_manager = SqlAlchemyManager(config)

print(f'SQLAlchemy engine: {sa_manager.engine.url}')

In [ ]:
# Прямой SQL-запрос через SQLAlchemy -> pandas DataFrame
df_tables = pd.read_sql_query(
    "SELECT tablename, pg_total_relation_size(quote_ident(tablename)) as size_bytes "
    "FROM pg_tables WHERE schemaname = 'public' ORDER BY size_bytes DESC LIMIT 15",
    con=sa_manager.engine
)

df_tables['size_mb'] = (df_tables['size_bytes'] / 1024 / 1024).round(2)
df_tables[['tablename', 'size_mb']]

In [ ]:
# Количество записей в ключевых таблицах
key_tables = [
    'auth_user',
    'cms_adp_userprofile',
    'cms_adp_menuaccesslog',
    'settings_auditlog',
    'core_messenger_message',
]

counts = []
for table in key_tables:
    try:
        result = pd.read_sql_query(f"SELECT COUNT(*) as cnt FROM {table}", con=sa_manager.engine)
        counts.append({'table': table, 'rows': result['cnt'].iloc[0]})
    except Exception:
        counts.append({'table': table, 'rows': 'N/A'})

pd.DataFrame(counts)

### SQLAlchemy -- параметризованные запросы

In [ ]:
from sqlalchemy import text

# Параметризованный запрос -- активность пользователей за последние N дней
days = 30

query = text("""
    SELECT u.username, u.last_login, u.date_joined,
           EXTRACT(DAY FROM NOW() - u.last_login) as days_since_login
    FROM auth_user u
    WHERE u.is_active = true
      AND u.last_login >= NOW() - INTERVAL ':days days'
    ORDER BY u.last_login DESC
""")

df_activity = pd.read_sql_query(query, con=sa_manager.engine, params={'days': days})
print(f'Активных за {days} дней: {len(df_activity)}')
df_activity.head(10)

## 3. Django raw SQL через QueryExecutor

Альтернативный способ -- использование `QueryExecutor` / `OrderedDictQueryExecutor` из ядра (работает через Django `connection.cursor()`).

In [ ]:
from src.core.utils.database import OrderedDictQueryExecutor

def get_audit_summary():
    return (
        "SELECT action, COUNT(*) as cnt "
        "FROM settings_auditlog "
        "GROUP BY action ORDER BY cnt DESC",
        ()
    )

try:
    result = OrderedDictQueryExecutor.fetchall(get_audit_summary)
    df_audit = pd.DataFrame(result)
    print('Аудит-лог по типам действий:')
    print(df_audit.to_string(index=False) if not df_audit.empty else 'Нет записей')
except Exception as e:
    print(f'Таблица auditlog не найдена или пуста: {e}')

## 4. Аналитика с pandas

In [ ]:
# Выгрузка всех пользователей в DataFrame для анализа
all_users = ErgoUser.objects.values(
    'public_id', 'username', 'email', 'is_active', 'is_staff',
    'is_superuser', 'date_joined', 'last_login'
)

df = pd.DataFrame(all_users)

if not df.empty:
    df['date_joined'] = pd.to_datetime(df['date_joined'])
    df['last_login'] = pd.to_datetime(df['last_login'])

    summary = {
        'Всего': len(df),
        'Активных': df['is_active'].sum(),
        'Неактивных': (~df['is_active']).sum(),
        'Staff': df['is_staff'].sum(),
        'Superuser': df['is_superuser'].sum(),
        'Без логина': df['last_login'].isna().sum(),
    }

    print('=== Сводка по пользователям ===')
    for k, v in summary.items():
        print(f'  {k}: {v}')
else:
    print('Нет пользователей в базе')

In [ ]:
# Когортный анализ: неделя регистрации vs последний вход
if not df.empty and df['last_login'].notna().any():
    cohort = df.copy()
    cohort['reg_week'] = cohort['date_joined'].dt.isocalendar().week
    cohort['login_week'] = cohort['last_login'].dt.isocalendar().week
    cohort['retention_weeks'] = (
        (cohort['last_login'] - cohort['date_joined']).dt.days / 7
    ).round(1)

    print('Retention (недели между регистрацией и последним входом):')
    print(cohort[['username', 'date_joined', 'last_login', 'retention_weeks']]
          .sort_values('retention_weeks', ascending=False)
          .head(10)
          .to_string(index=False))
else:
    print('Недостаточно данных для когортного анализа')

---

## Очистка ресурсов

In [ ]:
sa_manager.engine.dispose()
print('SQLAlchemy engine disposed. Done.')